In [2]:
from IPython import display

%pip install pandas

display.clear_output()

In [3]:
import os
import pandas as pd

In [4]:
root = "/Users/ellakennedy/Oyster-Video-Monitoring-System-2026/analysis/annotation_logs"

In [ ]:
def getIOU(box1, box2):
  box1X1 = max(box1["x1"], 0)
  box1Y1 = max(box1["y1"], 0)
  box2X1 = max(box2["x1"], 0)
  box2Y1 = max(box2["y1"], 0)

  intersectionX1 = max(box1X1, box2X1)
  intersectionY1 = max(box1Y1, box2Y1)
  intersectionX2 = min(box1["x2"], box2["x2"])
  intersectionY2 = min(box1["y2"], box2["y2"])

  if ((intersectionX2 < intersectionX1) or (intersectionY2 < intersectionY1)):
    return 0

  intersectionArea = (intersectionX2 - intersectionX1) * (intersectionY2 - intersectionY1)

  box1Area = (box1["x2"] - box1X1) * (box1["y2"] - box1Y1)

  box2Area = (box2["x2"] - box2X1) * (box2["y2"] - box2Y1)

  unionArea = box1Area + box2Area - intersectionArea

  return intersectionArea / unionArea


In [ ]:
true_positives = 0
total_positives = 0
total_oysters = 0

for i in range(1,19):
    subdir = root + "/video_" + str(i)
    detections = pd.read_csv(subdir + "/oyster_video_" + str(i) + "_log.csv")
    frames = pd.read_csv(subdir + "/video_" + str(i) + "_frames.csv")
    ground_truth = subdir + "/truth"

    for subdirs, dirs, files in os.walk(ground_truth):
        frame_num = 0
        for file in files:
            if (file != ".DS_Store"):
                true_frame = pd.read_csv(ground_truth + "/" + file)
                total_oysters += len(true_frame)

                detected_frame = detections[detections["frame"] == frames.iloc[frame_num, 0]]
                total_positives += len(detected_frame)

                for idx, row in detected_frame.iterrows():
                    #use iou to find potential match
                    box1X1 = float(detected_frame.at[row, "x1"])
                    box1Y1 = float(detected_frame.at[row, "y1"])
                    box1X2 = float(detected_frame.at[row, "x2"])
                    box1Y2 = float(detected_frame.at[row, "y2"])
                    box1 = {"y1":box1Y1, "x1":box1X1, "y2":box1Y2, "x2":box1X2}
                    bestIOU = 0
                    match = -1

                    for idx2, row2 in true_frame.iterrows():
                        box2X1 = (float(true_frame.at[row2, "xcenter"]) - true_frame.at[row2, "width"] / 2) * 640
                        box2Y1 = (float(true_frame.at[row2, "ycenter"]) - true_frame.at[row2, "height"] / 2) * 640
                        box2X2 = (float(true_frame.at[row2, "xcenter"]) + true_frame.at[row2, "width"] / 2) * 640
                        box2Y2 = (float(true_frame.at[row2, "ycenter"]) + true_frame.at[row2, "height"] / 2) * 640
                        box2 = {"y1":box2Y1, "x1":box2X1, "y2":box2Y2, "x2":box2X2}

                        iou = getIOU(box1, box2)

                        if iou > bestIOU:
                            bestIOU = iou
                            match =idx2

                    #check if detection is a true positive
                    if bestIOU >= 0.5 and match != -1:
                        if ((detected_frame.at[row, "smoothedLabel"] == "Oyster-Closed" and true_frame.at[match, "class"] == 0) or
                            (detected_frame.at[row, "smoothedLabel"] == "Oyster-Open" and true_frame.at[match, "class"] == 1)):
                            true_positives += 1

                frame_num += 1

#calculate precision
precision = true_positives / total_positives
print(f"precision: {precision}")

#calculate recall
recall = true_positives / total_oysters
print(f"recall: {recall}")

2
2
2
2
2
4
4
4
4
2
2
2
2
2
2
2
2
2
1
1
1
1
1
1
1
1
1
8
8
8
8
8
8
8
8
8
1
1
1
1
2
2
2
2
2
3
3
3
3
3
3
3
3
3
4
4
4
4
6
6
6
6
6
6
6
6
6
5
5
5
5
5
7
7
7
7
6
6
6
6
6
8
8
8
8
8
8
8
8
8
6
6
6
6
6
10
10
10
10
10
14
14
14
14
14
11
11
11
11
12
12
12
12
12
9
9
9
9
9
8
8
8
8
8
7
7
7
7
11
11
11
11
11
5
5
5
5
5
5
5
5
5
5
9
9
9
10
10
10
10
10
8
8
8
8
10
10
10
10
10
9
9
9
9
9
9
9
9
9
9
10
10
10
10
9
9
9
9
9
10
10
10
10
10
10
10
10
10
10
8
8
8
8
10
10
10
10
12
12
12
12
12
12
12
12
12
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
15
15
15
15
15
17
17
17
17
16
16
16
16
16
16
16
16
16
16
16
16
16
16
15
15
15
15
15
15
15
15
15
16
16
16
16
16
15
15
15
15
16
16
16
16
16
14
14
14
14
17
17
17
17
17
17
17
17
17
17
17
17
10
10
10
10
10
12
12
12
12
12
12
12
12
12
13
13
13
13
12
12
12
12
12
13
13
13
13
12
12
12
12
12
13
13
13
13
14
14
14
14
14
10
10
10
10
11
11
11
11
11
11
11
11
11
8
8
8
8
13
13
13
13
13
12
12
12
12
13
13
13
13
14
14
14
14
14
12
12
12
12
13
13
13
13
14
14
14
14
13
13
13
13
